# Convert Cleaned Data to Parquet

In [71]:
import duckdb

con = duckdb.connect()

con.execute("""
    COPY (
        SELECT *
        FROM read_csv_auto('Data/cleaned_accepted_rejected.csv', ignore_errors=false)
    ) TO 'Data/cleaned_accepted_rejected.parquet'
    (FORMAT PARQUET);
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

# Load Cleaned Data

In [72]:
import pandas as pd

df = pd.read_parquet('Data/cleaned_accepted_rejected.parquet')
df.head(5)

,loan_amnt,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,...,earliest_cr_line,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,application_type,mort_acc,pub_rec_bankruptcies
0,32000.0,60 months,10.49,687.65,B,B3,Public Service,10+ years,MORTGAGE,120000.0,...,Oct-1981,20.0,0.0,39687.0,57.8,42.0,w,Individual,2.0,0.0
1,4000.0,36 months,6.68,122.93,A,A3,System Analyst,4 years,MORTGAGE,83000.0,...,Sep-2003,16.0,0.0,1564.0,17.2,25.0,w,Individual,2.0,0.0
2,6025.0,36 months,10.91,197.00,B,B4,Admin assistant,10+ years,RENT,52000.0,...,Jun-2005,11.0,0.0,2706.0,12.8,25.0,w,Individual,0.0,0.0
3,25000.0,60 months,26.30,752.96,E,E5,Coordinator,10+ years,OWN,65000.0,...,Jul-1999,19.0,0.0,49461.0,24.7,33.0,w,Individual,0.0,0.0
4,20000.0,36 months,9.49,640.57,B,B2,Manager,10+ years,MORTGAGE,100000.0,...,Aug-1990,15.0,0.0,20440.0,56.3,32.0,w,Individual,3.0,0.0


In [73]:
df.shape

(1970154, 28)

In [74]:
print([column for column in df.columns if df[column].dtype == object])

['term', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'verification_status', 'issue_d', 'loan_status', 'purpose', 'title', 'zip_code', 'addr_state', 'earliest_cr_line', 'initial_list_status', 'application_type']


In [75]:
df.term.unique()

array([' 60 months', ' 36 months'], dtype=object)

grade & sub_grade are feature so drop it

In [76]:
df.drop('grade', axis=1, inplace=True)

In [77]:
dummies = ['sub_grade', 'verification_status', 'purpose', 'initial_list_status', 
           'application_type', 'home_ownership']
dummies_existing = [c for c in dummies if c in df.columns]
if dummies_existing:
    df = pd.get_dummies(df, columns=dummies_existing, drop_first=True)
df.head()

,loan_amnt,term,int_rate,installment,emp_title,emp_length,annual_inc,issue_d,loan_status,title,...,purpose_small_business,purpose_vacation,purpose_wedding,initial_list_status_w,application_type_Joint App,home_ownership_MORTGAGE,home_ownership_NONE,home_ownership_OTHER,home_ownership_OWN,home_ownership_RENT
0,32000.0,60 months,10.49,687.65,Public Service,10+ years,120000.0,Feb-2015,Current,Debt consolidation,...,False,False,False,True,False,True,False,False,False,False
1,4000.0,36 months,6.68,122.93,System Analyst,4 years,83000.0,Apr-2015,Fully Paid,Major purchase,...,False,False,False,True,False,True,False,False,False,False
2,6025.0,36 months,10.91,197.00,Admin assistant,10+ years,52000.0,Dec-2017,Fully Paid,Debt consolidation,...,False,False,False,True,False,False,False,False,False,True
3,25000.0,60 months,26.30,752.96,Coordinator,10+ years,65000.0,Feb-2018,Current,Debt consolidation,...,False,False,False,True,False,False,False,False,True,False
4,20000.0,36 months,9.49,640.57,Manager,10+ years,100000.0,Aug-2016,Fully Paid,Debt consolidation,...,False,False,False,True,False,True,False,False,False,False


## addr_state (address)
### feature engineer a zip code column from the address in the data set. Create a column called 'zip_code' that extracts the zip code from the address column.

In [78]:
df.addr_state.head()

0    CA
1    FL
2    MA
3    CA
4    NV
Name: addr_state, dtype: object

In [79]:
df['zip_code'] = df.addr_state.apply(lambda x: x[-5:])
df.zip_code.value_counts()

zip_code
CA    274024
TX    165451
NY    161664
FL    137790
IL     80188
NJ     72911
PA     67027
OH     65166
GA     64450
VA     55567
NC     54199
MI     50052
MD     47467
AZ     46227
MA     45666
CO     42711
WA     41188
MN     35394
IN     33415
CT     31385
MO     31289
TN     31245
NV     27669
WI     26175
SC     23782
AL     23371
OR     23002
LA     22536
KY     18858
OK     18049
KS     16801
AR     14485
UT     13302
MS     10986
NM     10232
NH      9867
HI      9491
RI      8783
WV      7057
NE      7028
DE      5558
MT      5486
DC      4698
AK      4651
VT      4337
ME      4318
WY      4224
SD      4028
ID      3657
ND      3245
IA         2
Name: count, dtype: int64

In [80]:
df = pd.get_dummies(df, columns=['zip_code'], drop_first=True)

In [81]:
df.drop('addr_state', axis=1, inplace=True)

## issue_d
### This would be data leakage, we wouldn't know beforehand whether or not a loan would be issued when using our model, so in theory we wouldn't have an issue_date, drop this feature.

In [82]:
df.drop('issue_d', axis=1, inplace=True)

In [83]:
w_p = df.loan_status.value_counts()[0] / df.shape[0]
w_n = df.loan_status.value_counts()[1] / df.shape[0]

print(f"Weight of positive values {w_p}")
print(f"Weight of negative values {w_n}")

Weight of positive values 0.4761774967845153
Weight of negative values 0.3923170472968103


C:\Users\User\AppData\Local\Temp\ipykernel_15584\4130237895.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  w_p = df.loan_status.value_counts()[0] / df.shape[0]
C:\Users\User\AppData\Local\Temp\ipykernel_15584\4130237895.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  w_n = df.loan_status.value_counts()[1] / df.shape[0]


In [84]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(df, test_size=0.15, random_state=168)

print(train.shape)
print(test.shape)

(1674630, 124)
(295524, 124)


# Removing Outliers

In [85]:
print(train[train['dti'] <= 50].shape)
print(train.shape)

(1666638, 124)
(1674630, 124)


In [86]:
print(train.shape)
train = train[train['annual_inc'] <= 250000]
train = train[train['dti'] <= 50]
train = train[train['open_acc'] <= 40]
train = train[train['total_acc'] <= 80]
train = train[train['revol_util'] <= 120]
train = train[train['revol_bal'] <= 250000]
print(train.shape)

(1674630, 124)
(1641858, 124)


## Normalizing the data

In [87]:
X_train, y_train = train.drop('loan_status', axis=1), train.loan_status
X_test, y_test = test.drop('loan_status', axis=1), test.loan_status

In [88]:
X_train.dtypes

loan_amnt      float64
term            object
int_rate       float64
installment    float64
emp_title       object
                ...   
zip_code_VT       bool
zip_code_WA       bool
zip_code_WI       bool
zip_code_WV       bool
zip_code_WY       bool
Length: 123, dtype: object

## Scaler

In [89]:
from sklearn.preprocessing import StandardScaler
import numpy as np

scaler = StandardScaler().set_output(transform="pandas")

X_train_num = X_train.select_dtypes(include=[np.number])
X_test_num = X_test.select_dtypes(include=[np.number])

# 3. Fit and transform (Executes in milliseconds)
x_train_scaled = scaler.fit_transform(X_train_num)
x_test_scaled = scaler.transform(X_test_num)

x_train_scaled

,loan_amnt,int_rate,installment,annual_inc,dti,open_acc,pub_rec,revol_bal,revol_util,total_acc,mort_acc,pub_rec_bankruptcies
1683280,-0.344929,0.594898,-0.591205,-0.657792,0.767801,-0.125006,1.433475,-0.181510,0.694163,1.434670,-0.815429,-0.347513
345357,1.411988,-0.335584,1.798224,0.607539,-0.769567,0.237792,-0.346945,0.076548,0.959369,0.667446,-0.815429,-0.347513
729157,-1.113581,0.181350,-1.048664,-1.194292,-2.023614,-1.757598,-0.346945,-0.881226,-1.815092,-1.378483,-0.282870,-0.347513
1588432,-0.564544,-1.448027,-0.542163,1.113672,-1.013410,0.781989,-0.346945,0.219399,-1.084756,-0.270271,-0.282870,-0.347513
815188,-0.454737,1.304132,-0.605812,-0.025126,-1.143459,-0.850603,1.433475,-0.292540,-0.150415,-0.526012,0.782248,2.405419
...,...,...,...,...,...,...,...,...,...,...,...,...
1076663,-1.113581,-0.815299,-1.092068,-0.748516,-0.900778,-1.213401,-0.346945,-0.655218,0.632962,-0.867001,-0.815429,-0.347513
1427757,-1.047696,1.004310,-0.929465,-0.455339,-0.458378,-0.125006,-0.346945,0.143853,1.306177,-0.270271,-0.282870,-0.347513
1453369,-0.344929,0.921600,-0.063361,0.835299,-0.385225,1.144788,-0.346945,-0.122332,-0.480902,1.264176,1.314808,-0.347513
1299711,-1.497907,0.189621,-1.501166,-1.290457,-0.243565,-0.306405,-0.346945,-0.852267,-1.851812,-0.526012,-0.815429,-0.347513


# Model Building

In [90]:
# remove Text month 
X_train['term'] = X_train['term'].astype(str).str.replace(' months', '').astype(float)
X_test['term'] = X_test['term'].astype(str).str.replace(' months', '').astype(float)

X_train_num = X_train.select_dtypes(include=[np.number])
X_test_num = X_test.select_dtypes(include=[np.number])

# 2. Convert the purely numeric data into float32 arrays for TensorFlow/Keras
X_train_array = np.array(X_train_num).astype(np.float32)
X_test_array = np.array(X_test_num).astype(np.float32)

## Artificial Neural Network

In [91]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [92]:
def print_score(true, pred, train=True):
    if train:
        clf_report = pd.DataFrame(classification_report(true, pred, output_dict=True))
        print("Train Result:\n================================================")
        print(f"Accuracy Score: {accuracy_score(true, pred) * 100:.2f}%")
        print("_______________________________________________")
        print(f"CLASSIFICATION REPORT:\n{clf_report}")
        print("_______________________________________________")
        print(f"Confusion Matrix: \n {confusion_matrix(true, pred)}\n")
        
    elif train==False:
        clf_report = pd.DataFrame(classification_report(true, pred, output_dict=True))
        print("Test Result:\n================================================")        
        print(f"Accuracy Score: {accuracy_score(true, pred) * 100:.2f}%")
        print("_______________________________________________")
        print(f"CLASSIFICATION REPORT:\n{clf_report}")
        print("_______________________________________________")
        print(f"Confusion Matrix: \n {confusion_matrix(true, pred)}\n")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import AUC
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score


def evaluate_nn(true, pred, train=True):
    label = 'Train' if train else 'Test'
    print(f'\n{label} Result  ================================================')
    print(f'Accuracy : {accuracy_score(true, pred)*100:.2f}%')
    print(pd.DataFrame(classification_report(true, pred, output_dict=True)))
    print(f'Confusion Matrix:\n{confusion_matrix(true, pred)}')


def plot_learning_evolution(r):
    auc_key = 'AUC' if 'AUC' in r.history else 'auc'
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    fig.suptitle('ANN - Lending Club Credit Risk', fontsize=14, fontweight='bold')

    ax1.plot(r.history['loss'],     label='Train', lw=2, color='steelblue')
    ax1.plot(r.history['val_loss'], label='Val',   lw=2, color='tomato', ls='--')
    ax1.set_title('Loss'); ax1.set_xlabel('Epoch')
    ax1.legend(); ax1.grid(alpha=0.3)

    ax2.plot(r.history[auc_key],          label='Train', lw=2, color='steelblue')
    ax2.plot(r.history[f'val_{auc_key}'], label='Val',   lw=2, color='tomato', ls='--')
    ax2.set_title('ROC-AUC'); ax2.set_xlabel('Epoch')
    ax2.legend(); ax2.grid(alpha=0.3)

    plt.tight_layout(); plt.show()


def build_model(n_features, hidden_units, dropout_rates, lr):
    inp = Input(shape=(n_features,))
    x = BatchNormalization()(inp)
    x = Dropout(dropout_rates[0])(x)
    for i, units in enumerate(hidden_units):
        x = Dense(units, activation='relu', kernel_initializer='he_normal',
                  kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
        x = BatchNormalization()(x)
        x = Dropout(dropout_rates[i + 1])(x)
    out = Dense(1, activation='sigmoid')(x)
    model = Model(inp, out)
    model.compile(optimizer=Adam(lr), loss='binary_crossentropy', metrics=[AUC(name='AUC')])
    return model


# Target encoding
bad_statuses = ['Charged Off', 'Default', 'Late (31-120 days)', 'Late (16-30 days)']
y_train_binary = np.isin(y_train, bad_statuses).astype(np.float32)
y_test_binary  = np.isin(y_test,  bad_statuses).astype(np.float32)

# Feature scaling (fix: original fed raw unscaled values -> model collapsed to majority class)
for _df in [X_train, X_test]:
    if _df['term'].dtype == object:
        _df['term'] = _df['term'].str.replace(' months', '', regex=False).astype(float)

X_train_num    = X_train.select_dtypes(include=[np.number])
X_test_num     = X_test.select_dtypes(include=[np.number])
scaler_ann     = StandardScaler()
X_train_scaled = scaler_ann.fit_transform(X_train_num).astype(np.float32)
X_test_scaled  = scaler_ann.transform(X_test_num).astype(np.float32)
print(f'Features: {X_train_scaled.shape[1]}  Train: {X_train_scaled.shape[0]:,}  Test: {X_test_scaled.shape[0]:,}')
print(f'Bad-loan rate: {y_train_binary.mean()*100:.1f}%')

# Class weights
neg, pos = float((y_train_binary == 0).sum()), float((y_train_binary == 1).sum())
total_n  = neg + pos
class_weights = {0: (1/neg)*(total_n/2), 1: (1/pos)*(total_n/2)}

# Architecture: 4-layer pyramid [512, 256, 128, 64] with decreasing dropout
hidden_units  = [512, 256, 128, 64]
dropout_rates = [0.05, 0.4, 0.3, 0.2, 0.1]  # len = 5 = 4 layers + 1

model = build_model(
    n_features    = X_train_scaled.shape[1],
    hidden_units  = hidden_units,
    dropout_rates = dropout_rates,
    lr            = 3e-4,
)
model.summary()

# Callbacks
early_stop = EarlyStopping(monitor='val_AUC', mode='max', patience=6,
                            restore_best_weights=True, verbose=1)
reduce_lr  = ReduceLROnPlateau(monitor='val_AUC', mode='max', factor=0.5,
                                patience=3, min_lr=1e-6, verbose=1)

# Train
r = model.fit(
    X_train_scaled, y_train_binary,
    validation_data = (X_test_scaled, y_test_binary),
    epochs          = 60,
    batch_size      = 4096,
    callbacks       = [early_stop, reduce_lr],
    class_weight    = class_weights,
    verbose         = 1,
)

plot_learning_evolution(r)

In [ ]:
y_train_prob = model.predict(X_train_scaled, batch_size=4096).flatten()
y_test_prob  = model.predict(X_test_scaled,  batch_size=4096).flatten()

threshold    = 0.5
y_train_pred = (y_train_prob >= threshold).astype(int)
y_test_pred  = (y_test_prob  >= threshold).astype(int)

evaluate_nn(y_train_binary.astype(int), y_train_pred, train=True)
evaluate_nn(y_test_binary.astype(int),  y_test_pred,  train=False)

train_auc = roc_auc_score(y_train_binary, y_train_prob)
test_auc  = roc_auc_score(y_test_binary,  y_test_prob)
print(f'ROC-AUC -> Train: {train_auc:.4f}  |  Test: {test_auc:.4f}')

# Best threshold by bad-loan F1
from sklearn.metrics import f1_score
thresholds = np.arange(0.10, 0.90, 0.05)
f1s  = [f1_score(y_test_binary.astype(int), (y_test_prob >= t).astype(int), pos_label=1) for t in thresholds]
best_t = thresholds[int(np.argmax(f1s))]
print(f'Best threshold (bad-loan F1): {best_t:.2f}  F1={max(f1s):.4f}')